# Notebook 10 — Final Evaluation

Produces a reproducible final report containing:
- model versions
- dataset source/version notes
- split strategy
- metrics
- thresholds
- error analysis
- limitations

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

OUTPUT_DIR = "D:/data viewer/data/processed"
REPORT_DIR = "D:/data viewer/reports"

## Model Versions

In [ ]:
model_versions = {
    'entity_matching': {
        'logistic_regression': 'v1.0 (trained on synthetic canonical master)',
        'random_forest': 'v1.0 (trained on synthetic canonical master)',
    },
    'anomaly_detection': {
        'isolation_forest': 'v1.0 (contamination=0.05)',
        'rolling_mad': 'v1.0 (window=96, threshold=3.5)',
    },
    'ai_reviewer': {
        'rule_based_classifier': 'v1.0 (deterministic rules on pairwise features)',
    },
}
print(json.dumps(model_versions, indent=2))

## Dataset Sources

In [ ]:
datasets = {
    'solar_power_generation': {
        'source': 'Kaggle (anikannal/solar-power-generation-data)',
        'license': 'Verify Kaggle terms',
        'rows': 139937,
        'usage': 'Anomaly detection, time-series analysis',
    },
    'febrl4': {
        'source': 'Python Record Linkage Toolkit',
        'license': 'Public domain (synthetic benchmark)',
        'rows': 10000,
        'usage': 'Entity resolution benchmark',
    },
    'canonical_master': {
        'source': 'Synthetic (generated in Phase 3)',
        'license': 'N/A',
        'rows': 500,
        'usage': 'Domain modeling, source simulation',
    },
}
print(json.dumps(datasets, indent=2))

## Split Strategy

In [ ]:
split_strategy = {
    'method': 'Entity-level split (train/validation/test)',
    'ratio': '70/15/15',
    'stratification': 'By label (match/non-match)',
    'note': 'Same entity does not appear in both train and test sets',
}
print(json.dumps(split_strategy, indent=2))

## Metrics

In [ ]:
features = pd.read_csv(f"{OUTPUT_DIR}/pairwise_features.csv")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X = features[['customer_similarity', 'installer_similarity', 'address_similarity', 'city_match', 'state_match', 'capacity_diff', 'status_match']].fillna(0)
y = features['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

metrics = {
    'logistic_regression': {
        'precision': round(precision_score(y_test, y_pred_lr), 3),
        'recall': round(recall_score(y_test, y_pred_lr), 3),
        'f1': round(f1_score(y_test, y_pred_lr), 3),
        'roc_auc': round(roc_auc_score(y_test, y_prob_lr), 3),
    },
    'random_forest': {
        'precision': round(precision_score(y_test, y_pred_rf), 3),
        'recall': round(recall_score(y_test, y_pred_rf), 3),
        'f1': round(f1_score(y_test, y_pred_rf), 3),
        'roc_auc': round(roc_auc_score(y_test, y_prob_rf), 3),
    },
}
print(json.dumps(metrics, indent=2))

## Thresholds

In [ ]:
thresholds = {
    'auto_match': 0.95,
    'manual_review': 0.50,
    'non_match': 0.0,
    'note': 'Thresholds calibrated on validation set. Adjust based on business risk tolerance.',
}
print(json.dumps(thresholds, indent=2))

## Error Analysis

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Logistic Regression Confusion Matrix:")
print(cm_lr)
print(f"\nFalse positives: {cm_lr[0,1]}")
print(f"False negatives: {cm_lr[1,0]}")

print("\nRandom Forest Confusion Matrix:")
print(cm_rf)
print(f"\nFalse positives: {cm_rf[0,1]}")
print(f"False negatives: {cm_rf[1,0]}")

## Limitations

In [ ]:
limitations = [
    'Synthetic canonical master data (not real operational data)',
    'No real solar-domain entity resolution ground truth',
    'FEBRL4 is synthetic, not solar-domain',
    'Epoch 2026 dataset not available (competition ended)',
    'ML models trained on limited synthetic features',
    'High recall but moderate precision (many false positives)',
]
for i, lim in enumerate(limitations, 1):
    print(f"{i}. {lim}")

## Save Final Report

In [ ]:
final_report = {
    'timestamp': datetime.now().isoformat(),
    'model_versions': model_versions,
    'dataset_sources': datasets,
    'split_strategy': split_strategy,
    'metrics': metrics,
    'thresholds': thresholds,
    'limitations': limitations,
}

with open(f"{REPORT_DIR}/final_evaluation_report.json", 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"\nFinal report saved to {REPORT_DIR}/final_evaluation_report.json")